# Phase 4 Stage 3 — v4-scratch v4 (RTX 5090, 32GB)

**v3 vs v2 — Root cause fix (mode collapse):**

| # | Item | v4-scratch v2 (failed) | v4-scratch v4 (this notebook) |
|---|---|---|---|
| 1 | `LORA_R / LORA_ALPHA` | 128 / 256 | **64 / 128** (clone English ref) |
| 2 | `USE_DORA` | True | **False** |
| 3 | `USE_RSLORA` | True → scale=22.6x | **False** → scale=2.0x (stable) |
| 4 | `LORA_DROPOUT` | 0.15 | **0.1** (English ref) |
| 5 | `WEIGHT_DECAY` | 0.01 | **0.001** (English ref & v3) |
| 6 | `NUM_EPOCHS` | 2 | **3** |
| 7 | `PER_DEVICE_BATCH` | 24 (eff=96) | **32 (eff=128)** — smoke first |
| 8 | `NEFTune alpha` | 5 | **5** (giữ) |
| 9 | Cell structure | monolithic build_model() | **split: tokenizer / base model / LoRA** |

**Root cause v2:** RSLoRA scale=22.6x gây gradient spike step 200 → mode collapse (model chỉ output "199").  
**Fix:** Standard LoRA r=64/alpha=128 (scale=2.0x), no DoRA, wd=0.001 — clone English ref config đã proven.

**Hardware:** RTX 5090 32GB | Target RMSLE 0.36–0.40

In [ ]:
# Chay neu chua co trong env:
#!uv add "transformers>=5.2.0" peft trl bitsandbytes accelerate datasets python-dotenv huggingface-hub matplotlib

In [ ]:
import os, re, sys, gc, json, time, math, random, shutil, tempfile
import numpy as np
from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login, snapshot_download, HfApi
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    EarlyStoppingCallback, PreTrainedTokenizerBase,
)
from trl import SFTTrainer, SFTConfig

@dataclass
class DataCollatorForCompletionOnlyLM:
    """Manual impl: trl.DataCollatorForCompletionOnlyLM removed in TRL 0.24.0."""
    response_template: List[int]
    tokenizer: PreTrainedTokenizerBase
    ignore_index: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids_list = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        max_len = max(len(x) for x in input_ids_list)
        bs = len(input_ids_list)
        padded = torch.full((bs, max_len), self.tokenizer.pad_token_id, dtype=torch.long)
        attn   = torch.zeros((bs, max_len), dtype=torch.long)
        labels = torch.full((bs, max_len), self.ignore_index, dtype=torch.long)
        tpl, tpl_len = self.response_template, len(self.response_template)
        for i, ids in enumerate(input_ids_list):
            n = len(ids)
            padded[i, :n] = ids
            attn[i, :n]   = 1
            for j in range(n - tpl_len, -1, -1):
                if ids[j : j + tpl_len].tolist() == tpl:
                    labels[i, j + tpl_len : n] = ids[j + tpl_len : n]
                    break
        return {"input_ids": padded, "attention_mask": attn, "labels": labels}

NOTEBOOK_DIR = Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from utils.evaluator import compute_metrics
from utils.rmsle_callback import RMSLEEvalCallback

print("Imports OK")
import transformers, peft, trl
for pkg, mod in [("torch", torch), ("transformers", transformers), ("peft", peft), ("trl", trl)]:
    print(f"  {pkg:<14}: {mod.__version__}")
print(f"NOTEBOOK_DIR: {NOTEBOOK_DIR}")

In [ ]:
# ── v4-scratch v4 constants ───────────────────────────────────────
# Config: clone English ref (r=64/alpha=128/no-DoRA/no-RSLoRA/wd=0.001)
# + NEFTune alpha=5 + data 269K + 3 epochs

BASE_MODEL   = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME = "SeanSunny/items_prompts_tv_4"

# Sequence
MAX_SEQ_LENGTH  = 208
MAX_NEW_TOKENS  = 4
QUESTION_PREFIX = "Sản phẩm này có giá bao nhiêu ?\n"
PRICE_PREFIX    = "\n\nGiá là: "

# LoRA — standard (NO DoRA, NO RSLoRA) → scale = alpha/r = 2.0x
LORA_R              = 64
LORA_ALPHA          = 128          # = 2 × r, English ref rule
LORA_DROPOUT        = 0.1          # English ref
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]
USE_DORA            = False        # bỏ — root cause collapse
USE_RSLORA          = False        # bỏ — scale 22.6x quá lớn

# Training
TRAIN_SIZE          = None
VAL_CALLBACK_SIZE   = 500
VAL_FULL_SIZE       = None
PLOT_SIZE           = 200
NUM_EPOCHS          = 3
PER_DEVICE_BATCH    = 32           # smoke first; fallback 24 nếu VRAM > 28GB
GRAD_ACCUM          = 4            # eff_batch = 128
LEARNING_RATE       = 2e-4
LR_SCHEDULER        = "cosine"
WARMUP_RATIO        = 0.03
WEIGHT_DECAY        = 0.001        # revert về English ref / v3 level
MAX_GRAD_NORM       = 0.3
OPTIM               = "paged_adamw_32bit"
NEFTUNE_ALPHA       = 5
GRADIENT_CHECKPOINTING = True
GROUP_BY_LENGTH     = True
EVAL_STEPS          = 500
SAVE_STEPS          = 500
SAVE_TOTAL_LIMIT    = 3
EARLY_STOP_PATIENCE = 3
LOGGING_STEPS       = 50
SEED                = 42

# Smoke
SMOKE_STEPS          = 30
SMOKE_VRAM_LIMIT_GB  = 28.0   # PyTorch limit; fallback PER_DEVICE_BATCH=24 nếu vượt

# Inference safety
PRED_CLAMP_MIN = 5
PRED_CLAMP_MAX = 1000
PARSE_REGEX    = r"[-+]?\d*\.\d+|\d+"

# Paths
ADAPTER_DIR     = NOTEBOOK_DIR / "weights" / "v4_scratch_v4_adapter"
RESULTS_FILE    = NOTEBOOK_DIR / "results" / "v4_scratch_v4_results.json"
PREDS_FILE      = NOTEBOOK_DIR / "results" / "v4_scratch_v4_val_predictions.json"
CHARTS_DIR      = NOTEBOOK_DIR / "results" / "charts"
HF_REPO_ADAPTER = "SeanSunny/qwen3.5-4b-vn-pricer-v4-scratch-v4"
HF_CKPT_BRANCH  = "last-checkpoint"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"BASE_MODEL   : {BASE_MODEL}")
print(f"DATASET_NAME : {DATASET_NAME}")
print(f"LoRA         : r={LORA_R}, alpha={LORA_ALPHA} (scale={LORA_ALPHA/LORA_R:.1f}x), "
      f"dropout={LORA_DROPOUT}, DoRA={USE_DORA}, RSLoRA={USE_RSLORA}")
print(f"Training     : epochs={NUM_EPOCHS}, eff_batch={PER_DEVICE_BATCH*GRAD_ACCUM}, "
      f"lr={LEARNING_RATE}, wd={WEIGHT_DECAY}, neftune={NEFTUNE_ALPHA}")
print(f"Smoke limit  : {SMOKE_VRAM_LIMIT_GB} GB (PyTorch)")
print(f"HF push      : {HF_REPO_ADAPTER}, branch={HF_CKPT_BRANCH}")

In [ ]:
# GPU check + HF login + dirs
assert torch.cuda.is_available(), "GPU khong kha dung."
gpu_name  = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cap = torch.cuda.get_device_capability()
print(f"GPU  : {gpu_name}")
print(f"VRAM : {total_vram:.1f} GB")
print(f"bf16 : {'yes' if cap[0] >= 8 else 'no'} (compute cap {cap})")

env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print(f"HF login OK (from {env_path})")
else:
    print(f"HF_TOKEN not found in {env_path}, prompting login")
    login()

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
(NOTEBOOK_DIR / "results").mkdir(exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dirs ready: weights/{ADAPTER_DIR.name} | results/ | results/charts/")

## 1. Spot-check dataset (269K train)

Hiển thị 10 random sample từ `items_prompts_tv_4` train để xác nhận format.  
In đầy đủ: **prompt** (gửi vào model) và **completion** (label model cần predict).

In [ ]:
ds_check = load_dataset(DATASET_NAME, split="train")
print(f"Dataset: {DATASET_NAME}")
print(f"  train : {len(ds_check):,} rows")
print(f"  cols  : {ds_check.column_names}\n")

rng = random.Random(SEED + 99)
sample_indices = rng.sample(range(len(ds_check)), 10)

for k, i in enumerate(sample_indices):
    row = ds_check[i]
    print(f"{'='*60}")
    print(f"Sample {k+1:2d}  idx={i}  |  completion={row['completion']!r}  "
          f"|  price={row['price_vnd_true']:>8,} VND")
    print(f"{'─'*60}")
    print("PROMPT (input to model):")
    print(row['prompt'])
    print(f"COMPLETION (label): {row['completion']!r}")
    print()

del ds_check
gc.collect()
print("Spot-check done. ds_check deleted.")

## 2. Token re-profile (5K sample từ 269K)

Verify `MAX_SUMMARY_TOKENS` cho `MAX_SEQ_LENGTH=208`.  
Truncation rate phải dưới 1%.

In [ ]:
tok_probe = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
ds_probe  = load_dataset(DATASET_NAME, split="train")
probe_idx = random.Random(SEED).sample(range(len(ds_probe)), 5000)
EOS = tok_probe.eos_token or "<|endoftext|>"

prompt_lens, full_lens = [], []
for i in probe_idx:
    row = ds_probe[i]
    pl = len(tok_probe.encode(row['prompt'], add_special_tokens=False))
    fl = len(tok_probe.encode(
        row['prompt'] + str(row['completion']) + "\n" + EOS,
        add_special_tokens=False))
    prompt_lens.append(pl)
    full_lens.append(fl)

prompt_lens = np.array(prompt_lens)
full_lens   = np.array(full_lens)

print(f"{'':15} {'Prompt':>8}  {'Full':>8}")
print(f"{'─'*35}")
for q in [50, 90, 95, 99]:
    print(f"  p{q:<3}        : {int(np.percentile(prompt_lens,q)):>8}  {int(np.percentile(full_lens,q)):>8}")
print(f"  max         : {int(prompt_lens.max()):>8}  {int(full_lens.max()):>8}")
print(f"  mean        : {prompt_lens.mean():>8.1f}  {full_lens.mean():>8.1f}")

print()
for L in [192, 208, 224, 256]:
    trunc = (full_lens > L).sum() / len(full_lens) * 100
    marker = "  <── chosen" if L == MAX_SEQ_LENGTH else ""
    print(f"  trunc @ max_seq_len={L:3d}: {trunc:.2f}%{marker}")

q_ids = tok_probe.encode(QUESTION_PREFIX, add_special_tokens=False)
p_ids = tok_probe.encode(PRICE_PREFIX,    add_special_tokens=False)
TOKENS_FIXED       = len(q_ids) + len(p_ids)
MAX_SUMMARY_TOKENS = MAX_SEQ_LENGTH - TOKENS_FIXED - MAX_NEW_TOKENS - 1 - 2
print(f"\nTOKENS_FIXED       : {TOKENS_FIXED}  (QUESTION={len(q_ids)}, PRICE={len(p_ids)})")
print(f"MAX_SUMMARY_TOKENS : {MAX_SUMMARY_TOKENS}")

del tok_probe, ds_probe, prompt_lens, full_lens
gc.collect()
print("Re-profile done. tok_probe + ds_probe deleted.")

## 3. Build model — 3 cells riêng biệt

Chia thành 3 cell để dễ inspect từng bước:
- **3a**: Load Tokenizer — in EOS / PAD token
- **3b**: Load Base Model 4-bit (NF4) — in memory footprint + architecture
- **3c**: Add LoRA adapter — in trainable params

`build_model()` gọi lại cả 3 sub-function khi cần (smoke → del → rebuild fresh).

In [ ]:
# Sub-functions cho build_model() — định nghĩa 1 lần, gọi nhiều lần

def _build_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "right"
    if tokenizer.eos_token_id is None:
        tokenizer.eos_token = "<|endoftext|>"
    return tokenizer

def _build_base_model(tokenizer):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
    )
    base_model = prepare_model_for_kbit_training(
        base_model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING
    )
    # Fix conv1d dtype (Qwen3.5 hybrid — lesson từ v1)
    for _m in base_model.modules():
        if isinstance(_m, torch.nn.Conv1d):
            _m.to(torch.bfloat16)
    return base_model

def _add_lora(base_model):
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        use_dora=USE_DORA,
        use_rslora=USE_RSLORA,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.enable_input_require_grads()
    return model

def build_model():
    """Build tokenizer + base 4-bit + LoRA. Called twice: smoke + full train."""
    tokenizer  = _build_tokenizer()
    base_model = _build_base_model(tokenizer)
    model      = _add_lora(base_model)
    return model, tokenizer

print("Sub-functions defined: _build_tokenizer / _build_base_model / _add_lora / build_model")

## 3a. Load Tokenizer

In [ ]:
# Load the Tokenizer
tokenizer = _build_tokenizer()

print(f"Tokenizer    : {BASE_MODEL}")
print(f"Vocab size   : {tokenizer.vocab_size:,}")
print(f"EOS token    : {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")
print(f"PAD token    : {tokenizer.pad_token!r}  (id={tokenizer.pad_token_id})")
print(f"Padding side : {tokenizer.padding_side}")

# Show tokenization of PRICE_PREFIX
price_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
print(f"\nPRICE_PREFIX tokenization:")
print(f"  ids    : {price_ids}")
print(f"  decoded: {tokenizer.decode(price_ids)!r}")

## 3b. Load Base Model (4-bit NF4)

In [ ]:
# Load the Base Model using 4-bit NF4 quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
base_model = prepare_model_for_kbit_training(
    base_model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING
)
for _m in base_model.modules():
    if isinstance(_m, torch.nn.Conv1d):
        _m.to(torch.bfloat16)

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.2f} GB")
print(f"\nModel architecture (first decoder layer):")
print(base_model.model.layers[0])

## 3c. Add LoRA Adapter

In [ ]:
# Add LoRA adapter (standard — no DoRA, no RSLoRA)
model = _add_lora(base_model)
model.print_trainable_parameters()

# Show LoRA matrices inside first attention layer
first_attn = model.model.layers[0].self_attn
print(f"\nLoRA inside layers[0].self_attn.q_proj:")
print(f"  lora_A: {list(first_attn.q_proj.lora_A.values())[0].weight.shape}")
print(f"  lora_B: {list(first_attn.q_proj.lora_B.values())[0].weight.shape}")
print(f"  scaling: alpha/r = {LORA_ALPHA}/{LORA_R} = {LORA_ALPHA/LORA_R:.1f}x  (vs RSLoRA 22.6x trong v2)")
print(f"\nMemory footprint (base + adapter): {model.get_memory_footprint() / 1e9:.2f} GB")

## 4. Dataset preprocess + tokenize

- Cắt summary ở token level nếu dài hơn `MAX_SUMMARY_TOKENS`
- Format: `QUESTION_PREFIX + summary + PRICE_PREFIX + completion + \n + EOS`
- In sample để xác nhận format trước khi tokenize

In [ ]:
ds = load_dataset(DATASET_NAME)
print(f"Splits: train={len(ds['train']):,} | val={len(ds['val']):,} | test={len(ds['test']):,}")

q_ids = tokenizer.encode(QUESTION_PREFIX, add_special_tokens=False)
p_ids = tokenizer.encode(PRICE_PREFIX,    add_special_tokens=False)
TOKENS_FIXED       = len(q_ids) + len(p_ids)
MAX_SUMMARY_TOKENS = MAX_SEQ_LENGTH - TOKENS_FIXED - MAX_NEW_TOKENS - 1 - 2
print(f"TOKENS_FIXED       : {TOKENS_FIXED}")
print(f"MAX_SUMMARY_TOKENS : {MAX_SUMMARY_TOKENS}")

train_full_raw  = ds["train"].shuffle(seed=SEED)
if TRAIN_SIZE is not None:
    train_full_raw = train_full_raw.select(range(TRAIN_SIZE))
val_full_raw    = ds["val"]
val_callback_raw = ds["val"].shuffle(seed=SEED).select(range(VAL_CALLBACK_SIZE))

def preprocess(example):
    p       = example["prompt"]
    summary = p[len(QUESTION_PREFIX):-len(PRICE_PREFIX)]
    s_ids   = tokenizer.encode(summary, add_special_tokens=False)
    if len(s_ids) > MAX_SUMMARY_TOKENS:
        s_ids   = s_ids[:MAX_SUMMARY_TOKENS]
        summary = tokenizer.decode(s_ids, skip_special_tokens=True).rstrip()
    full_text = (QUESTION_PREFIX + summary + PRICE_PREFIX
                 + example["completion"] + "\n" + tokenizer.eos_token)
    return {"text": full_text}

train_ds         = train_full_raw.map(preprocess,   desc="Preprocess train")
val_ds_for_trainer = val_callback_raw.map(preprocess, desc="Preprocess val (trainer)")

def tokenize_fn(example):
    return tokenizer(example["text"], truncation=True,
                     max_length=MAX_SEQ_LENGTH, padding=False)

train_ds_tok = train_ds.map(tokenize_fn, batched=False, remove_columns=["text"])
val_ds_tok   = val_ds_for_trainer.map(tokenize_fn, batched=False, remove_columns=["text"])
train_ds_tok = train_ds_tok.map(lambda ex: {"length": len(ex["input_ids"])}, desc="Compute length")

print(f"\ntrain_ds_tok : {len(train_ds_tok):,} rows")
print(f"val_ds_tok   : {len(val_ds_tok):,} rows")
print(f"Sample lengths: {[len(train_ds_tok[i]['input_ids']) for i in range(5)]}")

# Show 1 sample full text để confirm format
print(f"\nSample 0 full text:")
print(repr(train_ds[0]['text']))

## 5. DataCollator + Mask verify + RMSLEEvalCallback

In [ ]:
response_template_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
decoded_back = tokenizer.decode(response_template_ids)
print(f"response_template_ids : {response_template_ids}")
print(f"decoded back          : {decoded_back!r}")
assert decoded_back == PRICE_PREFIX, f"Mismatch: {decoded_back!r} != {PRICE_PREFIX!r}"

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

# Mask verify — chỉ completion tokens được học, phần còn lại = -100
sample_text = train_ds[0]["text"]
tok_out = tokenizer(sample_text, return_tensors="pt",
                    max_length=MAX_SEQ_LENGTH, truncation=True)
batch_in = [{"input_ids": tok_out["input_ids"][0].tolist(),
             "attention_mask": tok_out["attention_mask"][0].tolist()}]
labels = collator(batch_in)["labels"][0]
non_masked = labels[labels != -100]
if len(non_masked) == 0:
    raise RuntimeError("MASK VERIFY FAILED — không tìm thấy PRICE_PREFIX trong sample!")
print(f"\nMask verify:")
print(f"  Non-masked tokens : {non_masked.tolist()}")
print(f"  Decoded           : {tokenizer.decode(non_masked.tolist(), skip_special_tokens=False)!r}")
print(f"  Expected comp     : {train_full_raw[0]['completion']!r}")
print("  MASK VERIFY PASS")

# RMSLEEvalCallback
rmsle_callback = RMSLEEvalCallback(
    tokenizer=tokenizer,
    val_subset=val_callback_raw,
    max_new_tokens=MAX_NEW_TOKENS,
    clamp_min=PRED_CLAMP_MIN,
    clamp_max=PRED_CLAMP_MAX,
)
print(f"\nRMSLEEvalCallback ready (n={len(val_callback_raw)} val subset)")

## 6. Smoke VRAM test (30 steps)

**Mục đích:** Xác nhận VRAM với `PER_DEVICE_BATCH=32` (eff_batch=128) trước khi commit ~20h training.  
**Abort:** Nếu peak > **28 GB** (PyTorch) → đổi `PER_DEVICE_BATCH=24`, restart kernel, re-run.  
**Sau smoke:** `del smoke_trainer, model, tokenizer` → `gc.collect()` → `empty_cache()` → rebuild fresh.

In [ ]:
torch.cuda.reset_peak_memory_stats()
SMOKE_DIR = tempfile.mkdtemp(prefix="smoke_v3_")
print(f"Smoke dir: {SMOKE_DIR}")

smoke_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_tok.select(
        range(min(SMOKE_STEPS * PER_DEVICE_BATCH * GRAD_ACCUM, len(train_ds_tok)))),
    data_collator=collator,
    args=SFTConfig(
        output_dir=SMOKE_DIR,
        max_steps=SMOKE_STEPS,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        bf16=True,
        max_grad_norm=MAX_GRAD_NORM,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        train_sampling_strategy="group_by_length" if GROUP_BY_LENGTH else "random",
        length_column_name="length",
        neftune_noise_alpha=NEFTUNE_ALPHA,
        logging_steps=5,
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
        seed=SEED,
    ),
)
t0 = time.time()
smoke_trainer.train()
smoke_train_sec    = time.time() - t0
smoke_vram_peak    = torch.cuda.max_memory_allocated() / 1e9
sec_per_step       = smoke_train_sec / SMOKE_STEPS
total_steps_full   = math.ceil(len(train_ds_tok) / (PER_DEVICE_BATCH * GRAD_ACCUM)) * NUM_EPOCHS
total_time_est_hr  = total_steps_full * sec_per_step / 3600

print(f"\n{'='*45}")
print(f"SMOKE RESULT (batch={PER_DEVICE_BATCH}, eff={PER_DEVICE_BATCH*GRAD_ACCUM})")
print(f"{'='*45}")
print(f"VRAM peak (PyTorch) : {smoke_vram_peak:.2f} GB  (limit: {SMOKE_VRAM_LIMIT_GB} GB)")
print(f"Sec/step            : {sec_per_step:.2f}s")
print(f"Est. full train     : {total_steps_full:,} steps, ~{total_time_est_hr:.1f} hr")

# Test RMSLEEvalCallback (untrained — chỉ verify không crash)
print(f"\nTesting RMSLEEvalCallback ({VAL_CALLBACK_SIZE} subset)...")
t0 = time.time()
test_rmsle = rmsle_callback._compute_rmsle(model)
callback_sec = time.time() - t0
n_evals = total_steps_full // EVAL_STEPS
print(f"Smoke RMSLE    : {test_rmsle:.4f}  (untrained — verify only)")
print(f"Callback time  : {callback_sec:.1f}s per eval")
print(f"Total eval cost: ~{n_evals * callback_sec / 60:.0f} min ({n_evals} evals)")

if smoke_vram_peak > SMOKE_VRAM_LIMIT_GB:
    raise RuntimeError(
        f"VRAM {smoke_vram_peak:.2f}GB > {SMOKE_VRAM_LIMIT_GB}GB limit!\n"
        f"Action: set PER_DEVICE_BATCH=24 in constants cell, restart kernel, re-run."
    )
print(f"\nSMOKE PASS — VRAM OK")

# ── Cleanup smoke (QUAN TRỌNG: xóa trước rebuild) ─────────────────
del smoke_trainer, model, tokenizer
gc.collect()
torch.cuda.empty_cache()
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
print(f"Cleanup done. VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 7. Resume detection + Rebuild fresh model

**Strategy (vast.ai disconnect protection):**
1. Check local `weights/v4_scratch_v4_adapter/checkpoint-*`
2. Nếu không → check HF Hub branch `last-checkpoint`
3. Rebuild fresh model (smoke đã pollute model state)

In [ ]:
RESUME_PATH = None

# 1. Check local checkpoints
local_ckpts = sorted(ADAPTER_DIR.glob("checkpoint-*"))
if local_ckpts:
    RESUME_PATH = str(local_ckpts[-1])
    print(f"[Resume] Local: {RESUME_PATH}")
else:
    # 2. Check HF Hub last-checkpoint branch
    api = HfApi()
    try:
        refs = api.list_repo_refs(HF_REPO_ADAPTER, repo_type="model")
        if HF_CKPT_BRANCH in [b.name for b in refs.branches]:
            print(f"[Resume] HF branch '{HF_CKPT_BRANCH}' found, downloading...")
            RESUME_PATH = snapshot_download(
                repo_id=HF_REPO_ADAPTER,
                revision=HF_CKPT_BRANCH,
                local_dir=str(ADAPTER_DIR / "_hub_resume"),
            )
            print(f"[Resume] Downloaded: {RESUME_PATH}")
        else:
            print(f"[Resume] No '{HF_CKPT_BRANCH}' on HF — fresh start.")
    except Exception as e:
        print(f"[Resume] HF check failed (OK for first run): {type(e).__name__}: {e}")

if RESUME_PATH is None:
    print("[Resume] Fresh start from step 0.")
else:
    print(f"[Resume] Will resume from: {RESUME_PATH}")

# 3. Rebuild fresh model (smoke đã xong)
print("\n[Rebuild] Building fresh model...")
model, tokenizer = build_model()
model.print_trainable_parameters()
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

# Re-init callback với tokenizer mới
rmsle_callback = RMSLEEvalCallback(
    tokenizer=tokenizer,
    val_subset=val_callback_raw,
    max_new_tokens=MAX_NEW_TOKENS,
    clamp_min=PRED_CLAMP_MIN,
    clamp_max=PRED_CLAMP_MAX,
)
print("RMSLEEvalCallback re-initialized.")

## 8. Full train — Standard LoRA + NEFTune + best by RMSLE + HF live push

Config (clone English ref + NEFTune):
- `r=64, alpha=128` (scale=2.0x — stable, no collapse)
- `NEFTune alpha=5` — noise on embeddings, +1-3% generative RMSLE
- `metric_for_best_model="eval_rmsle"` — best ckpt theo RMSLE thực, không theo CE loss
- `hub_strategy="checkpoint"` → push mỗi 500 step → resume-able nếu vast.ai disconnect

In [ ]:
torch.cuda.reset_peak_memory_stats()
t_train_start = time.time()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_tok,
    eval_dataset=val_ds_tok,
    data_collator=collator,
    callbacks=[
        rmsle_callback,
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
    ],
    args=SFTConfig(
        output_dir=str(ADAPTER_DIR),
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        optim=OPTIM,
        bf16=True,
        max_grad_norm=MAX_GRAD_NORM,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        train_sampling_strategy="group_by_length" if GROUP_BY_LENGTH else "random",
        length_column_name="length",
        neftune_noise_alpha=NEFTUNE_ALPHA,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        load_best_model_at_end=True,
        metric_for_best_model="eval_rmsle",
        greater_is_better=False,
        logging_steps=LOGGING_STEPS,
        report_to="none",
        seed=SEED,
        push_to_hub=True,
        hub_model_id=HF_REPO_ADAPTER,
        hub_strategy="checkpoint",
        hub_private_repo=True,
    ),
)

if RESUME_PATH:
    print(f"[Train] Resuming from {RESUME_PATH}")
    trainer.train(resume_from_checkpoint=RESUME_PATH)
else:
    print("[Train] Fresh start")
    trainer.train()

trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

total_train_sec = time.time() - t_train_start
final_vram_peak = torch.cuda.max_memory_allocated() / 1e9
print(f"\nTraining done: {total_train_sec/60:.1f} min ({total_train_sec/3600:.2f} hr)"
      f" | VRAM peak: {final_vram_peak:.2f} GB")

log_history      = trainer.state.log_history
train_losses     = [(int(e["step"]), float(e["loss"]))        for e in log_history if "loss"        in e and "eval_loss" not in e]
eval_losses      = [(int(e["step"]), float(e["eval_loss"]))   for e in log_history if "eval_loss"   in e]
eval_rmsle_curve = [(int(e["step"]), float(e["eval_rmsle"])) for e in log_history if "eval_rmsle" in e]
lr_curve         = [(int(e["step"]), float(e["learning_rate"])) for e in log_history if "learning_rate" in e]

print(f"\nLog: train_loss={len(train_losses)} | eval_ce={len(eval_losses)} "
      f"| eval_rmsle={len(eval_rmsle_curve)} | lr={len(lr_curve)}")

best_ckpt_step = best_ckpt_rmsle = best_ckpt_path = None
best_ckpt_path = trainer.state.best_model_checkpoint
if eval_rmsle_curve:
    best_ckpt_step, best_ckpt_rmsle = min(eval_rmsle_curve, key=lambda x: x[1])
    print(f"Best eval_rmsle (subset n={VAL_CALLBACK_SIZE}): {best_ckpt_rmsle:.4f} @ step {best_ckpt_step}")
    print(f"Best ckpt path: {best_ckpt_path}")

# ── Cleanup trainer sau khi extract logs ──────────────────────────
del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 9. Final generative eval — full 3,926 val (best ckpt đã auto-load)

In [ ]:
model.eval()
for _m in model.modules():
    if isinstance(_m, torch.nn.Conv1d):
        _m.to(torch.bfloat16)

def predict_one(prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m   = re.search(PARSE_REGEX, gen)
    if m:
        pk = max(PRED_CLAMP_MIN, min(int(float(m.group())), PRED_CLAMP_MAX))
    else:
        pk = 0
    return pk, gen

val_for_eval = val_full_raw if VAL_FULL_SIZE is None else val_full_raw.select(range(VAL_FULL_SIZE))
preds_vnd, trues_vnd, raw_outs = [], [], []
clamp_count = 0
t_eval_start = time.time()

for item in tqdm(val_for_eval, desc="Generative eval (full val)"):
    pk, raw = predict_one(item["prompt"])
    preds_vnd.append(pk * 1000)
    trues_vnd.append(item["price_vnd_true"])
    raw_outs.append(raw)
    if pk in (PRED_CLAMP_MIN, PRED_CLAMP_MAX):
        clamp_count += 1

t_eval = time.time() - t_eval_start
metrics_final = compute_metrics(
    np.array(trues_vnd, dtype=float), np.array(preds_vnd, dtype=float))
n_eval = len(val_for_eval)

print("=" * 55)
print(f"v4-scratch v4 — {n_eval} val (best ckpt by eval_rmsle)")
print("=" * 55)
print(f"RMSLE  : {metrics_final['rmsle']:.4f}  (primary)")
print(f"MAE    : {metrics_final['mae']:>12,.0f} VND")
print(f"MAPE   : {metrics_final['mape']:.1f}%")
print(f"R2     : {metrics_final['r2']:.2f}%")
print(f"Zero preds    : {preds_vnd.count(0)}")
print(f"Clamp trigger : {clamp_count}")
print(f"Sec/item      : {t_eval/n_eval:.2f}s")
print("─" * 55)
print(f"v3 ref        : RMSLE=0.4426 | MAE=80,100 VND")
print(f"Day4 v8 ref   : RMSLE=0.4004 | MAE=79,853 VND")
print(f"Target        : RMSLE 0.36-0.40")
print("=" * 55)

# Show 20 sample predictions
print(f"\nSample predictions (20 random):")
rng_s = random.Random(SEED + 1)
show_idx = rng_s.sample(range(n_eval), 20)
for rank, i in enumerate(show_idx):
    tv, pv = trues_vnd[i], preds_vnd[i]
    err = abs(pv - tv) / tv * 100 if tv > 0 else 0
    flag = "OK" if err < 20 else ("~" if err < 60 else "!!")
    print(f"  [{flag}] true={tv:>8,} pred={pv:>8,}  err={err:5.1f}%  raw={raw_outs[i]!r}")

In [ ]:
# Save val predictions (input cho 07_ensemble.ipynb)
preds_dump = [
    {"idx": i, "pred_vnd": int(preds_vnd[i]), "true_vnd": int(trues_vnd[i])}
    for i in range(n_eval)
]
with open(PREDS_FILE, "w", encoding="utf-8") as f:
    json.dump(preds_dump, f, ensure_ascii=False)
print(f"Saved predictions ({n_eval} rows): {PREDS_FILE}")

samples_out = []
for i in range(min(20, n_eval)):
    tv, pv = trues_vnd[i], preds_vnd[i]
    samples_out.append({
        "idx": i,
        "prompt_excerpt": val_for_eval[i]["prompt"][:120],
        "generated_raw": raw_outs[i],
        "pred_vnd": pv, "true_vnd": tv,
        "error_pct": round(abs(pv-tv)/tv*100, 1) if tv > 0 else None,
    })

results = {
    "version": "v4_scratch_v4",
    "model": BASE_MODEL,
    "dataset": DATASET_NAME,
    "config": {
        "train_size": len(train_ds_tok),
        "val_callback_size": VAL_CALLBACK_SIZE,
        "val_full_size": n_eval,
        "max_seq_length": MAX_SEQ_LENGTH,
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "use_dora": USE_DORA, "use_rslora": USE_RSLORA,
        "target_modules": LORA_TARGET_MODULES,
        "num_epochs": NUM_EPOCHS,
        "per_device_batch": PER_DEVICE_BATCH, "grad_accum": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE, "lr_scheduler": LR_SCHEDULER,
        "warmup_ratio": WARMUP_RATIO, "weight_decay": WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM, "optim": OPTIM,
        "neftune_alpha": NEFTUNE_ALPHA, "group_by_length": GROUP_BY_LENGTH,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "early_stop_patience": EARLY_STOP_PATIENCE,
    },
    "smoke": {
        "vram_peak_gb": round(smoke_vram_peak, 2),
        "sec_per_step": round(sec_per_step, 2),
        "total_steps_est": total_steps_full,
        "total_time_est_hr": round(total_time_est_hr, 1),
    },
    "best_ckpt": {
        "step": best_ckpt_step,
        "rmsle_subset": round(best_ckpt_rmsle, 4) if best_ckpt_rmsle else None,
        "path": str(best_ckpt_path),
    } if best_ckpt_step else None,
    "vram_train_peak_gb": round(final_vram_peak, 2),
    "total_train_sec": round(total_train_sec, 1),
    "sec_per_val_item": round(t_eval / n_eval, 2),
    "train_loss_curve": train_losses,
    "eval_loss_curve": eval_losses,
    "eval_rmsle_curve": eval_rmsle_curve,
    "lr_curve": lr_curve,
    "metrics_final": metrics_final,
    "samples_20": samples_out,
}
with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved results             : {RESULTS_FILE}")

## 10. Charts (8 PNG → `results/charts/`)

1. Train CE loss (raw + EMA)
2. Eval CE loss
3. Eval RMSLE + best step marker
4. Learning rate schedule
5. 2×2 dashboard
6. Pred vs True scatter (200 sample, log scale)
7. Error % histogram
8. Per-price-bucket RMSLE

In [ ]:
PREFIX = "v4_scratch_v4"

def ema_smooth(values, alpha=0.1):
    if not values: return values
    out = [values[0]]
    for v in values[1:]:
        out.append(alpha * v + (1-alpha) * out[-1])
    return out

plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150})

# Chart 1: Train CE loss
if train_losses:
    steps_t, vals_t = zip(*train_losses)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_t, vals_t, alpha=0.3, color="C0", label="raw")
    ax.plot(steps_t, ema_smooth(list(vals_t)), color="C0", lw=2, label="EMA")
    ax.set(xlabel="Step", ylabel="Train CE Loss",
           title=f"v4-scratch v4 — Train CE Loss")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(CHARTS_DIR / f"{PREFIX}_01_train_loss.png"); plt.show()

# Chart 2: Eval CE loss
if eval_losses:
    steps_e, vals_e = zip(*eval_losses)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_e, vals_e, "o-", color="C1", lw=2)
    ax.set(xlabel="Step", ylabel="Eval CE Loss",
           title="v4-scratch v4 — Eval CE Loss (overfit detection)")
    ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(CHARTS_DIR / f"{PREFIX}_02_eval_loss.png"); plt.show()

# Chart 3: Eval RMSLE + best step
if eval_rmsle_curve:
    steps_r, vals_r = zip(*eval_rmsle_curve)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_r, vals_r, "o-", color="C2", lw=2)
    if best_ckpt_step:
        ax.axvline(best_ckpt_step, color="red", ls="--", alpha=0.7,
                   label=f"Best step {best_ckpt_step} (RMSLE={best_ckpt_rmsle:.4f})")
        ax.legend()
    ax.axhline(0.4004, color="grey", ls=":", alpha=0.5, label="v8 ref 0.4004")
    ax.set(xlabel="Step", ylabel="Eval RMSLE (500 subset)",
           title="v4-scratch v4 — Eval RMSLE (primary metric)")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(CHARTS_DIR / f"{PREFIX}_03_eval_rmsle.png"); plt.show()

# Chart 4: LR
if lr_curve:
    steps_l, vals_l = zip(*lr_curve)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_l, vals_l, color="C3", lw=2)
    ax.set(xlabel="Step", ylabel="Learning Rate",
           title=f"v4-scratch v4 — LR cosine (warmup={WARMUP_RATIO})")
    ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(CHARTS_DIR / f"{PREFIX}_04_lr.png"); plt.show()

# Chart 5: 2×2 dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
if train_losses:
    ax = axes[0,0]; steps_t, vals_t = zip(*train_losses)
    ax.plot(steps_t, vals_t, alpha=0.3, color="C0")
    ax.plot(steps_t, ema_smooth(list(vals_t)), color="C0", lw=2)
    ax.set_title("Train CE Loss"); ax.set_xlabel("Step"); ax.grid(alpha=0.3)
if eval_losses:
    ax = axes[0,1]; steps_e, vals_e = zip(*eval_losses)
    ax.plot(steps_e, vals_e, "o-", color="C1", lw=2)
    ax.set_title("Eval CE Loss"); ax.set_xlabel("Step"); ax.grid(alpha=0.3)
if eval_rmsle_curve:
    ax = axes[1,0]; steps_r, vals_r = zip(*eval_rmsle_curve)
    ax.plot(steps_r, vals_r, "o-", color="C2", lw=2)
    if best_ckpt_step:
        ax.axvline(best_ckpt_step, color="red", ls="--", alpha=0.7,
                   label=f"Best@{best_ckpt_step}"); ax.legend()
    ax.set_title("Eval RMSLE"); ax.set_xlabel("Step"); ax.grid(alpha=0.3)
if lr_curve:
    ax = axes[1,1]; steps_l, vals_l = zip(*lr_curve)
    ax.plot(steps_l, vals_l, color="C3", lw=2)
    ax.set_title("Learning Rate"); ax.set_xlabel("Step"); ax.grid(alpha=0.3)
fig.suptitle(f"v4-scratch v4 Dashboard (RMSLE={metrics_final['rmsle']:.4f})", fontsize=13)
fig.tight_layout(); fig.savefig(CHARTS_DIR / f"{PREFIX}_05_dashboard.png"); plt.show()
print(f"Charts 1-5 saved to {CHARTS_DIR}")

In [ ]:
rng_c = np.random.default_rng(SEED)
plot_idx  = rng_c.choice(n_eval, size=min(PLOT_SIZE, n_eval), replace=False)
trues_plot = np.array([trues_vnd[i] for i in plot_idx], dtype=float)
preds_plot = np.array([preds_vnd[i] for i in plot_idx], dtype=float)

# Chart 6: Scatter pred vs true (log scale)
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(trues_plot, preds_plot, alpha=0.5, s=20)
mn = min(trues_plot.min(), preds_plot[preds_plot > 0].min() if (preds_plot > 0).any() else 1)
mx = max(trues_plot.max(), preds_plot.max())
ax.plot([mn, mx], [mn, mx], "r--", alpha=0.7, label="y=x (perfect)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set(xlabel="True price (VND, log)", ylabel="Predicted price (VND, log)",
       title=f"v4-scratch v4 — Pred vs True ({PLOT_SIZE} sample)\nRMSLE={metrics_final['rmsle']:.4f}")
ax.legend(); ax.grid(alpha=0.3, which="both")
fig.tight_layout(); fig.savefig(CHARTS_DIR / f"{PREFIX}_06_scatter_200.png"); plt.show()

# Chart 7: Error histogram
errors_pct = np.abs(preds_plot - trues_plot) / np.maximum(trues_plot, 1) * 100
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.clip(errors_pct, 0, 200), bins=40, color="C4", edgecolor="black", alpha=0.7)
ax.axvline(np.median(errors_pct), color="red",    ls="--", label=f"Median {np.median(errors_pct):.1f}%")
ax.axvline(np.mean(errors_pct),   color="orange", ls="--", label=f"Mean {np.mean(errors_pct):.1f}%")
ax.set(xlabel="Abs error % (clip@200%)", ylabel="Count",
       title=f"v4-scratch v4 — Error distribution ({PLOT_SIZE} sample)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(CHARTS_DIR / f"{PREFIX}_07_error_hist.png"); plt.show()
print("Charts 6-7 saved")

In [ ]:
# Chart 8: Per-price-bucket RMSLE
trues_arr = np.array(trues_vnd, dtype=float)
preds_arr = np.array(preds_vnd, dtype=float)
buckets = [
    ("<50K",    0,       50_000),
    ("50-100K", 50_000,  100_000),
    ("100-200K",100_000, 200_000),
    ("200-500K",200_000, 500_000),
    ("500K-1M", 500_000, 1_000_001),
]
labels, rmsles, counts = [], [], []
for name, lo, hi in buckets:
    mask = (trues_arr >= lo) & (trues_arr < hi)
    n = mask.sum()
    if n == 0: continue
    r = float(np.sqrt(np.mean((np.log1p(preds_arr[mask]) - np.log1p(trues_arr[mask]))**2)))
    labels.append(name); rmsles.append(r); counts.append(int(n))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, rmsles, color="C5", edgecolor="black", alpha=0.8)
for bar, r, n in zip(bars, rmsles, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{r:.3f}\n(n={n})", ha="center", va="bottom", fontsize=9)
ax.axhline(metrics_final['rmsle'], color="red", ls="--", alpha=0.7,
           label=f"Overall RMSLE: {metrics_final['rmsle']:.4f}")
ax.axhline(0.4004, color="grey", ls=":", alpha=0.6, label="v8 ref: 0.4004")
ax.set(xlabel="Price bucket (VND)", ylabel="RMSLE",
       title=f"v4-scratch v4 — RMSLE by price bucket (full {n_eval} val)")
ax.legend(); ax.grid(alpha=0.3, axis="y")
fig.tight_layout(); fig.savefig(CHARTS_DIR / f"{PREFIX}_08_bucket_rmsle.png"); plt.show()

print(f"\nAll 8 charts saved to {CHARTS_DIR}:")
for p in sorted(CHARTS_DIR.glob(f"{PREFIX}_*.png")):
    print(f"  {p.name}")

## 11. Push HF main branch (best model)

In [ ]:
print(f"Pushing best model to {HF_REPO_ADAPTER} (main, private)...")
model.push_to_hub(HF_REPO_ADAPTER, private=True)
tokenizer.push_to_hub(HF_REPO_ADAPTER, private=True)
print(f"Pushed: https://huggingface.co/{HF_REPO_ADAPTER}")

print("\n" + "="*60)
print("v4-scratch v4 — FINAL SUMMARY")
print("="*60)
print(f"Config           : r={LORA_R}, alpha={LORA_ALPHA}, scale={LORA_ALPHA/LORA_R:.1f}x, "
      f"no-DoRA, no-RSLoRA, wd={WEIGHT_DECAY}")
print(f"Train size       : {len(train_ds_tok):,}")
print(f"Epochs           : {NUM_EPOCHS}")
print(f"Train time       : {total_train_sec/3600:.2f} hr")
print(f"VRAM peak (smoke): {smoke_vram_peak:.2f} GB")
print(f"VRAM peak (full) : {final_vram_peak:.2f} GB")
if best_ckpt_step:
    print(f"Best ckpt step   : {best_ckpt_step} (subset RMSLE={best_ckpt_rmsle:.4f})")
print(f"Final RMSLE      : {metrics_final['rmsle']:.4f}")
print(f"Final MAE        : {metrics_final['mae']:,.0f} VND")
print(f"Final R2         : {metrics_final['r2']:.2f}%")
print(f"v3 ref           : 0.4426 | v8 ref: 0.4004 | Target: 0.36-0.40")
print("="*60)

gc.collect(); torch.cuda.empty_cache()

## 12. Log template — `phase2_execution_log.md` Run #5

Sau khi chạy xong, paste kết quả vào `phase2_execution_log.md` mục **Run #5 — v4-scratch v4**:

```markdown
## Run #5 — v4-scratch v4 — YYYY-MM-DD

**Notebook:** `fine_tune_qwen/06_train_v4_scratch_v4.ipynb`
**Hardware:** NVIDIA RTX 5090 32GB (vast.ai, $0.388/hr)
**Python env:** torch X | transformers X | peft X | trl X

### Config (vs v4-scratch-v2 — root cause fix)
- r=64 / alpha=128 (scale=2.0x) — NO DoRA, NO RSLoRA
- weight_decay=0.001, dropout=0.1, 3 epochs, NEFTune alpha=5
- Data: 269K (items_prompts_tv_4), eff_batch=128

### Smoke result
- VRAM peak  : X.XX GB (limit 28 GB)
- Sec/step   : X.XXs
- Est. total : N steps, X.X hr

### Full train
- Total steps: N
- Wall-clock : X.XX hr | VRAM peak: X.XX GB
- Best ckpt  : step N, eval_rmsle (subset 500) = X.XXXX
- HF ckpt   : https://huggingface.co/SeanSunny/qwen3.5-4b-vn-pricer-v4-scratch-v4/tree/last-checkpoint
- HF main   : https://huggingface.co/SeanSunny/qwen3.5-4b-vn-pricer-v4-scratch-v4

### Final eval (3,926 val)
| Metric | Value | vs v3 | vs v8 |
|---|---|---|---|
| RMSLE | X.XXXX | +/- X.XXXX | +/- X.XXXX |
| MAE   | XX,XXX VND | | |
| MAPE  | XX.X% | | |
| R2    | X.XXXX | | |

### Issues / Deviations
- (ghi nếu có)

### Notes for ensemble (07_ensemble.ipynb)
- val_predictions: `results/v4_scratch_v4_val_predictions.json` (n=3,926)
- Ensemble = v3 + v4-scratch-v4 + v8 (3-model Ridge log-space)
```